# Advanced Time-Frequency Analysis

## Part of the open_dvm Toolbox

Building on the foundational TFR techniques from `03_tfr_analysis.ipynb`, this tutorial explores advanced time-frequency analysis methods: wavelet parameter selection, alternative decomposition methods, and multi-subject statistical testing.

## Learning Objectives

After completing this tutorial, you will:

- **Understand wavelet parameter trade-offs** — How cycle ranges affect temporal vs. frequency resolution
- **Compare decomposition methods** — Morlet wavelets vs. Hilbert transform
- **Aggregate effects across subjects** — Build grand-average time-frequency results
- **Apply statistical testing** — FDR-corrected significance testing on time-frequency data

**Prerequisites:** Complete `03_tfr_analysis.ipynb` first to understand basic TFR computation, baseline correction, and lateralization analysis.

## Overview

### Key Steps

1. **Wavelet parameter exploration** — How temporal-frequency resolution trade-offs affect your results
2. **Wavelet vs. Hilbert comparison** — When to use alternative decomposition methods
3. **Multi-subject condition comparisons** — Aggregating across subjects and testing statistical significance

## Section 1: Setup and Configuration

### 1.1 Import Required Libraries

In [ ]:
# Enable inline plotting and suppress warnings
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from IPython.display import display

# Import analysis tools
import warnings
warnings.filterwarnings('ignore')

from open_dvm.analysis import TFR
from open_dvm.support.FolderStructure import FolderStructure
from open_dvm.visualization.plot import plot_tfr_timecourse

print("✓ All imports successful!")

### 1.2 Load Preprocessed Data with Eye-Tracking Quality Control

In [ ]:
# Download (if not already cached) and locate the preprocessed tutorial
# dataset -- a fast-path that skips running 01_preprocessing.ipynb yourself
from open_dvm.support.datasets import fetch_processed_data

project_folder = fetch_processed_data()
os.chdir(project_folder)

# Subject number (1-7 in this dataset)
sj = 2

# Eye-tracking quality control
eye_dict = {
    'use_tracker': True,        # Enable eye-tracking exclusion
    'window_oi': (0, 0.3),      # Window: 0-300 ms post-stimulus
    'angle_thresh': 1,          # Threshold: 1 degree visual angle
    'viewing_dist': 70,         # Viewing distance (cm)
    'screen_res': (1920, 1080), # Screen resolution (pixels)
    'screen_h': 29,             # Screen height (cm)
    'drift_correct': (-0.2, 0)  # Drift correction window
}

# Load preprocessed data
df, epochs = FolderStructure().load_processed_epochs(
    sj=sj,                       # subject ID
    fname='ses_01_main',         # preprocessed file name (sub_<sj>_ses_01_main-epo.fif)
    preproc_name='main',         # preprocessing pipeline name (locates parameter files)
    eye_dict=eye_dict            # eye-tracking exclusion criteria
)

print(f'✓ Subject {sj} data loaded')
print(f'  • {len(epochs)} trials')
print(f'  • {epochs.info["nchan"]} channels')
print(f'  • Sampling rate: {epochs.info["sfreq"]} Hz')

---

## Section 2: Wavelet Parameter Exploration

**Research Question**: How do wavelet parameters affect the temporal-frequency resolution trade-off? Do more cycles improve frequency resolution while sacrificing temporal resolution?

**Key insight**: The number of Morlet wavelet cycles determines resolution:
- **Fewer cycles** (3-8) — Better temporal precision but noisier frequency estimates
- **Standard** (3-10) — Balanced trade-off (default recommendation)
- **More cycles** (4-12) — Better frequency selectivity but smeared in time

**Why this matters**: For questions requiring precise temporal localization (e.g., event-related components), fewer cycles are better. For frequency-specific hypotheses (e.g., narrowband oscillations), more cycles improve selectivity.

In [ ]:
# Try different wavelet cycle ranges
cycle_configs = {
    'Fast (3-8)': (3, 8),
    'Standard (3-10)': (3, 10),
    'Slow (4-12)': (4, 12)
}

tfr_cycles = {}

print('Computing TFR with different wavelet cycle ranges...')

for config_name, cycles in cycle_configs.items():
    tfr_temp = TFR(
        sj=sj,
        epochs=epochs,
        df=df,
        min_freq=4,                # Start from theta band (4 Hz)
        max_freq=40,               # Up to low gamma (40 Hz)
        num_frex=25,               # 25 frequencies between 4-40 Hz
        cycle_range=cycles,        # Current cycle configuration under comparison (loop variable)
        freq_scaling='log',        # Logarithmic frequency spacing
        baseline=(-0.2, 0),        # Baseline period: -200 to 0 ms pre-stimulus
        base_method='trial_spec',  # Trial-specific baseline correction
        downsample=256             # Downsample to 256 Hz
    )
    
    tfr_result = tfr_temp.condition_tfrs(
        pos_labels=None,                      # No position-based lateralization for this comparison
        cnds={'block_type': ['localizer']},   # Localizer trials only
        elec_oi='posterior',                  # Occipital/parietal electrodes (see select_electrodes)
        window_oi=(-0.2, 0.5),                # Analysis window: -200 to 500 ms
    )
    
    # Store the full per-config result object (not just a summary array) so
    # the comparison plot below can show each config's own data, not a
    # stale reference to whichever config ran last
    tfr_cycles[config_name] = tfr_result['localizer']
    print(f'  ✓ cycles {cycles}')

print('\n✓ TFR computation with different cycle ranges complete')

In [ ]:
# Compare wavelet cycle ranges using built-in plotting
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plt.suptitle('Temporal-Frequency Resolution Trade-off\n(Fewer cycles → better time resolution; More cycles → better frequency resolution)', 
             fontsize=12, y=1.00)

for idx, (config_name, tfr_obj) in enumerate(tfr_cycles.items()):
    plt.sca(axes[idx])
    
    # Use built-in plotting function
    plot_tfr_timecourse(
        tfr={config_name: tfr_obj},
        elec_oi=list(tfr_obj.ch_names), # All electrodes used in the computation above
        timecourse='2d',                # Full time-frequency heatmap
        contour=True,                   # Filled contour lines instead of a raw heatmap
        levels=20,                      # Number of contour levels
        cmap='viridis',                 # Sequential colormap (raw power, not a difference)
        vmin=-3,                        # Fixed color scale (with vmax below) for direct
        vmax=3,                         # comparison across the three cycle configs
        onset_times=[0]                 # Mark stimulus onset
    )
    
    plt.title(f'Cycles: {config_name}', fontsize=11, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

print("✓ Wavelet cycle comparison complete")
print("\nKey observations:")
print("  • Fast (3-8): Sharper temporal boundaries, noisier frequency estimates")
print("  • Standard (3-10): Balanced approach (recommended default)")
print("  • Slow (4-12): Better frequency separation, broader temporal smearing")

---

## Section 3: Wavelet vs. Hilbert Comparison

**Research Question**: When should you use alternative time-frequency decomposition methods? How do Morlet wavelets compare to Hilbert transforms?

**Context**:
- **Morlet wavelets** — Standard in TFR; provides good time-frequency localization with controlled trade-offs
- **Hilbert transform** — Faster; internally bandpass-filters each frequency band before transforming; provides instantaneous amplitude/phase
- **Multi-taper methods** — Optimal for static spectra; less useful for event-related TFR

**Approach**: The `TFR` class supports both decomposition methods directly -- switching `.method` from `'wavelet'` to `'hilbert'` on an existing `TFR` instance and calling `condition_tfrs()` again recomputes the decomposition with the new method, without constructing a fresh object. The cell below computes both on the same main-task frontal-electrode data and compares them side by side.

In [ ]:
# Morlet wavelet TFR computation
tfr_morlet_obj = TFR(
    sj=sj,
    epochs=epochs,
    df=df,
    min_freq=4,                # Start from theta band (4 Hz)
    max_freq=40,               # Up to low gamma (40 Hz)
    num_frex=25,               # 25 frequencies between 4-40 Hz
    cycle_range=(3, 10),       # Standard wavelet cycles (balanced resolution trade-off)
    freq_scaling='log',        # Logarithmic frequency spacing
    baseline=(-0.2, 0),        # Baseline period: -200 to 0 ms pre-stimulus
    method='wavelet',          # Morlet wavelet decomposition (compared against Hilbert below)
    base_method='cnd_spec',    # Condition-specific baseline correction
    downsample=256             # Downsample to 256 Hz
)

tfr_morlet = tfr_morlet_obj.condition_tfrs(
    pos_labels=None,                       # No position-based lateralization for this comparison
    cnds={'block_type': ['main']},         # Main task trials only
    elec_oi=['Fp1', 'Fp2', 'AF3', 'AF4'],  # Frontal electrodes
    window_oi=(-0.2, 0.5),                 # Analysis window: -200 to 500 ms
)

print("✓ Morlet (Wavelet) TFR computed")
print(f"  • Data shape: {tfr_morlet['main'].data.shape}")

# Reuse the same object for the Hilbert method rather than constructing a
# new TFR instance: changing `.method` after construction is safe here --
# frequency-band-dependent internals regenerate lazily from the new method
# the next time condition_tfrs() is called.
tfr_morlet_obj.method = 'hilbert'
tfr_hilbert = tfr_morlet_obj.condition_tfrs(
    pos_labels=None,                       # No position-based lateralization for this comparison
    cnds={'block_type': ['main']},         # Main task trials only
    elec_oi=['Fp1', 'Fp2', 'AF3', 'AF4'],  # Frontal electrodes
    window_oi=(-0.2, 0.5),                 # Analysis window: -200 to 500 ms
)

print("✓ Hilbert (Analytic Signal) TFR computed")
print(f"  • Data shape: {tfr_hilbert['main'].data.shape}")

In [ ]:
# Side-by-side comparison visualization using plot_tfr_timecourse
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plt.suptitle('Wavelet vs. Hilbert Transform: Time-Frequency Decomposition Comparison', 
             fontsize=12, fontweight='bold', y=1.02)

# Left subplot: Morlet (Wavelet)
plt.sca(axes[0])
plot_tfr_timecourse(
    tfr=tfr_morlet,
    elec_oi=['Fp1', 'Fp2', 'AF3', 'AF4'],  # Frontal electrodes used in the computation above
    timecourse='2d',    # Full time-frequency heatmap
    contour=True,       # Filled contour lines instead of a raw heatmap
    levels=20,          # Number of contour levels
    cmap='viridis',     # Sequential colormap (raw power, not a difference)
    vmin=0,             # Fixed color scale (with vmax below) for direct
    vmax=2,             
    onset_times=[0]     # Mark stimulus onset
)
axes[0].set_title('Morlet Wavelet\n(Good frequency selectivity)', fontsize=11, fontweight='bold')

# Right subplot: Hilbert (Analytic Signal)
plt.sca(axes[1])
plot_tfr_timecourse(
    tfr=tfr_hilbert,
    elec_oi=['Fp1', 'Fp2', 'AF3', 'AF4'],  # Frontal electrodes used in the computation above
    timecourse='2d',    # Full time-frequency heatmap
    contour=True,       # Filled contour lines instead of a raw heatmap
    levels=20,          # Number of contour levels
    cmap='viridis',     # Sequential colormap (raw power, not a difference)
    vmin=0,             # Fixed color scale (with vmax below) for direct
    vmax=2,             
    onset_times=[0]     # Mark stimulus onset
)
axes[1].set_title('Hilbert Transform\n(Sharp temporal boundaries)', fontsize=11, fontweight='bold')

plt.tight_layout()
display(fig)
plt.close()

print("✓ Method comparison visualization complete")

---

## Section 4: Multi-Subject Condition Comparisons with Statistical Testing

**Research Question**: Do condition effects on time-frequency signatures generalize across subjects? Where are effects statistically significant?

**Key concept**: Individual TFR patterns vary due to anatomy, habituation, and noise. To establish robust findings, aggregate effects across subjects and apply statistical tests.

**Approach**:
1. Loop over all 7 subjects, flagging main-task trials by display density: `T+D+` (target and distractor both present) vs. `T-D-` (both absent)
2. Compute and save each subject's TFR for both conditions over occipital electrodes (`condition_tfrs`, `f_name='display_density'`)
3. Load the saved per-subject results back in aggregated form (`FolderStructure().read_tfr(..., sjs='all')`)
4. Visualize grand-average spectrograms for both conditions, plus a 1D timecourse with FDR-corrected significance testing across subjects (`plot_tfr_timecourse(..., stats='fdr')`)

In [ ]:
for subject_id in range(1, 8):
    try:
        # Load data for this subject
        df_sj, epochs_sj = FolderStructure().load_processed_epochs(
            sj=subject_id,                # subject ID
            fname='ses_01_main',          # preprocessed file name (sub_<sj>_ses_01_main-epo.fif)
            preproc_name='main',          # preprocessing pipeline name (locates parameter files)
            eye_dict=eye_dict             # eye-tracking exclusion criteria
        )
        # Flag high-density (target+distractor present) vs low-density (both absent) displays
        df_sj.loc[(df_sj.target_presence != 'absent') & (df_sj.distractor_presence == 'absent'), 'display_type'] =  'T+D+'
        df_sj.loc[(df_sj.target_presence == 'absent') & (df_sj.distractor_presence == 'absent'), 'display_type'] =  'T-D-'

        sj_tfr = TFR(
            sj=subject_id,
            epochs=epochs_sj,
            df=df_sj,
            min_freq=4,                # Start from theta band (4 Hz)
            max_freq=40,               # Up to low gamma (40 Hz)
            num_frex=25,               # 25 frequencies between 4-40 Hz
            cycle_range=(3, 10),       # Standard wavelet cycles (balanced resolution trade-off)
            freq_scaling='log',        # Logarithmic frequency spacing
            baseline=(-0.2, 0),        # Baseline period: -200 to 0 ms pre-stimulus
            method='wavelet',          # Morlet wavelet decomposition
            base_method='cnd_spec',    # Condition-specific baseline correction
            downsample=256             # Downsample to 256 Hz
        )

        tfr_result = sj_tfr.condition_tfrs(
            pos_labels=None,                            # No position-based lateralization for this comparison
            cnds={'display_type': ['T+D+', 'T-D-']},    # High- vs low-density displays
            elec_oi=['PO3','PO4','O1','O2','Pz'],       # Occipital electrodes
            window_oi=(-0.2, 0.5),                      # Analysis window: -200 to 500 ms
            f_name='display_density'                    # Analysis identifier for saved output
        )
    except Exception as e:
        print(f'  ✗ Subject {subject_id} failed: {str(e)}')

print('\n✓ All subjects processed')

In [ ]:
# Load aggregated tfr results across all subjects
print('\nLoading tfr data...')
tfr_data = FolderStructure().read_tfr(
    tfr_folder_path=['wavelet'],   # Decomposition method used above
    tfr_name='display_density',    # Matches f_name used when saving above
    cnds = ['T+D+', 'T-D-'],       # Conditions to load (must match saved keys)
    sjs='all'                      # Load all subjects
)
print('✓ Data loaded')

In [ ]:
# Visualize condition effects: spectrograms + alpha timecourse
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(14, 10))
gs = gridspec.GridSpec(2, 2, height_ratios=[1, 0.7], hspace=0.35, wspace=0.3)
axes = [fig.add_subplot(gs[0, 0]), 
        fig.add_subplot(gs[0, 1]),
        fig.add_subplot(gs[1, :])]

plt.suptitle('Grand Average Condition Effects: T+D+ vs T-D-', 
             fontsize=13, fontweight='bold', y=0.98)

# ===== Top row: 2D spectrograms =====
conditions = ['T+D+', 'T-D-']
for idx, cnd in enumerate(conditions):
    plt.sca(axes[idx])
    plot_tfr_timecourse(
        tfr=tfr_data,
        cnds = [cnd],       # This subplot's condition only
        elec_oi=['PO3','PO4','O1','O2','Pz'],      # All electrodes used in the computation above
        timecourse='2d',    # Full time-frequency heatmap
        contour=True,       # Filled contour lines instead of a raw heatmap
        levels=20,          # Number of contour levels
        cmap='viridis',     # Sequential colormap (raw power, not a difference)
        vmin=None,          # Auto-scaled per subplot (conditions aren't directly
        vmax=None,          # compared here -- that's the bottom panel's job)
        onset_times=[0]     # Mark stimulus onset
    )
    axes[idx].set_title(f'Condition: {cnd}', fontsize=11, fontweight='bold')
    axes[idx].axhline(8, color='red', linestyle='--', linewidth=1.5, alpha=0.6, label='Alpha band')
    axes[idx].axhline(12, color='red', linestyle='--', linewidth=1.5, alpha=0.6)

# ===== Bottom row: 1D alpha timecourse =====
plt.sca(axes[2])
plot_tfr_timecourse(
    tfr=tfr_data,
    cnds = conditions,        # Both conditions, overlaid
    elec_oi=['PO3','PO4','O1','O2','Pz'],  # All electrodes used in the computation above
    freq_oi =(8, 12),         # Average within the alpha band marked above
    timecourse='1d',          # 1D timecourse (averaged over frequency)
    stats = 'fdr',            # FDR-corrected significance vs. zero, per condition
    cnd_diff=('T+D+', 'T-D-'),# Also test whether the two conditions differ
    onset_times=[0]           # Mark stimulus onset
)

plt.tight_layout()
display(fig)
plt.close()

print("✓ Grand average visualization complete")

---

## Section 5: Summary and Next Steps

### Section 2: Wavelet Parameter Exploration
- Wavelet **cycle range** controls the temporal-frequency resolution trade-off
- Fewer cycles (3-8): Better temporal precision for event-locked components
- More cycles (4-12): Better frequency selectivity for narrowband oscillations
- Standard (3-10) provides a good default balance

### Section 3: Method Comparison
- **Morlet wavelets**: Smooth spectral transitions, excellent frequency selectivity
- **Hilbert transform**: Sharper temporal boundaries, requires pre-filtering
- Choice depends on your research question and data characteristics
- The TFR class supports dynamic method switching for easy comparison

### Section 4: Multi-Subject Generalization
- Individual TFR patterns vary substantially across subjects (anatomy, habituation, task engagement)
- Grand average aggregation reveals robust population effects
- Statistical testing (FDR-corrected) identifies time-frequency regions with significant condition effects
- Spectrograms + 1D timecourse visualization effectively communicates both detail and summary

### Best Practices for TFR Analysis
1. **Always visualize**: Spectrograms reveal patterns that summary statistics miss
2. **Check assumptions**: Verify baseline periods, stationarity, and condition balance
3. **Consider multiple methods**: Wavelets, Hilbert, and other decompositions offer different perspectives
4. **Aggregate across subjects**: Single-subject findings may not generalize
5. **Report effect sizes**: p-values alone are insufficient; include Cohen's d, eta-squared, or similar
6. **Account for multiple comparisons**: FDR or Bonferroni correction is essential for time-frequency data

### Next Steps
- Explore **lateralization effects** using topo_flip (see `03_tfr_analysis.ipynb`)
- Implement **frequency-band differences** (e.g., alpha vs. theta)
- Apply **permutation testing** for robust non-parametric inference
- Examine **inter-subject variability** as a feature of individual differences
- Consider **source localization** (e.g., inverse models) if spatial resolution is critical